In [ ]:
!pip install timm facenet-pytorch opencv-python scikit-learn

In [ ]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
import timm
import cv2, os, gc
import numpy as np
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score

device = "cuda" if torch.cuda.is_available() else "cpu"

### **Dataset Prep**

In [ ]:
class DeepfakeDataset(Dataset):
    def __init__(self, root_dir, transform=None, limit=1500):
        self.paths, self.labels = [], []
        self.transform = transform

        for label, folder in enumerate(['real','fake']):
            folder_path = os.path.join(root_dir, folder)
            files = os.listdir(folder_path)[:limit]

            for img in files:
                self.paths.append(os.path.join(folder_path, img))
                self.labels.append(label)

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = cv2.imread(self.paths[idx])
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        if self.transform:
            img = self.transform(img)

        return img, self.labels[idx]

### **Transforms**

In [ ]:
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224,224)),
    transforms.ToTensor(),
])

### **DataLoader**

In [ ]:
dataset = DeepfakeDataset("/content/data", transform)
loader = DataLoader(dataset, batch_size=8, shuffle=True)  # small batch!

### **Memory Cleanup Function**

In [ ]:
def clear_memory():
    torch.cuda.empty_cache()
    gc.collect()

### **Training Function**

In [ ]:
def train_model(model, name):
    model = model.to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    loss_fn = nn.BCEWithLogitsLoss()

    for epoch in range(2):
        model.train()
        losses = []

        for x,y in loader:
            x = x.to(device)
            y = y.float().unsqueeze(1).to(device)

            pred = model(x)
            loss = loss_fn(pred, y)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            losses.append(loss.item())

        print(f"{name} Epoch {epoch}: {np.mean(losses)}")

    torch.save(model.state_dict(), f"/content/{name}.pth")

    return None

### **TRAIN MODELS**

**1. EfficientNet**

In [ ]:
model = timm.create_model('efficientnet_b0', pretrained=True, num_classes=1)
train_model(model, "eff")

del model
clear_memory()

#### **2. ResNet50**

In [ ]:
model = timm.create_model('resnet50', pretrained=True, num_classes=1)
train_model(model, "res")

del model
clear_memory()

**3. Vision Transformer**

In [ ]:
model = timm.create_model('vit_small_patch16_224', pretrained=True, num_classes=1)
train_model(model, "vit")

del model
clear_memory()

**4. FFT Model**

In [ ]:
class FFTModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3,16,3,padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1)
        )
        self.fc = nn.Linear(16,1)

    def forward(self,x):
        x = torch.fft.fft2(x).real
        x = self.conv(x).view(x.size(0),-1)
        return self.fc(x)

model = FFTModel()
train_model(model, "fft")

del model
clear_memory()

### **Evaluation Function**

In [ ]:
def evaluate(model):
    model.eval()
    preds, labels = [], []

    with torch.no_grad():
        for x,y in loader:
            x = x.to(device)
            out = torch.sigmoid(model(x)).cpu().numpy()

            preds.extend(out)
            labels.extend(y.numpy())

    preds_bin = [1 if p>0.5 else 0 for p in preds]

    return {
        "acc": accuracy_score(labels, preds_bin),
        "auc": roc_auc_score(labels, preds),
        "f1": f1_score(labels, preds_bin)
    }

### **LOADING MODELS FOR TEST**

**EfficientNet**

In [ ]:
model = timm.create_model('efficientnet_b0', pretrained=False, num_classes=1)
model.load_state_dict(torch.load("/content/eff.pth"))
model = model.to(device)

results["eff"] = evaluate(model)

del model
clear_memory()

**ResNet50**

In [ ]:
model = timm.create_model('resnet50', pretrained=False, num_classes=1)
model.load_state_dict(torch.load("/content/res.pth"))
model = model.to(device)

results["res"] = evaluate(model)

del model
clear_memory()

**Vision Transformer (ViT)**

In [ ]:
model = timm.create_model('vit_small_patch16_224', pretrained=False, num_classes=1)
model.load_state_dict(torch.load("/content/vit.pth"))
model = model.to(device)

results["vit"] = evaluate(model)

del model
clear_memory()

**FFT Model**

In [ ]:
class FFTModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3,16,3,padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1)
        )
        self.fc = nn.Linear(16,1)

    def forward(self,x):
        x = torch.fft.fft2(x).real
        x = self.conv(x).view(x.size(0),-1)
        return self.fc(x)

model = FFTModel()
model.load_state_dict(torch.load("/content/fft.pth"))
model = model.to(device)

results["fft"] = evaluate(model)

del model
clear_memory()

In [ ]:
for k,v in results.items():
    print(k, v)

**### FINAL ENSEMBLE**

### **Loading only top 2 models**

In [ ]:
model1 = timm.create_model('efficientnet_b0', pretrained=False, num_classes=1)
model1.load_state_dict(torch.load("/content/eff.pth"))
model1 = model1.to(device)

model2 = timm.create_model('vit_small_patch16_224', pretrained=False, num_classes=1)
model2.load_state_dict(torch.load("/content/vit.pth"))
model2 = model2.to(device)

### **Ensemble Prediction**

In [ ]:
def ensemble_eval():
    preds, labels = [], []

    with torch.no_grad():
        for x,y in loader:
            x = x.to(device)

            p1 = torch.sigmoid(model1(x))
            p2 = torch.sigmoid(model2(x))

            p = ((p1 + p2) / 2).cpu().numpy()

            preds.extend(p)
            labels.extend(y.numpy())

    preds_bin = [1 if p>0.5 else 0 for p in preds]

    return {
        "acc": accuracy_score(labels, preds_bin),
        "auc": roc_auc_score(labels, preds),
        "f1": f1_score(labels, preds_bin)
    }

### **Evaluate the Ensemble**

In [ ]:
ensemble_results = ensemble_eval()
print("Ensemble:", ensemble_results)

### **Weighted Ensemble**

In [ ]:
def ensemble_eval_weighted(w1=0.5, w2=0.5):
    preds, labels = [], []

    with torch.no_grad():
        for x,y in loader:
            x = x.to(device)

            p1 = torch.sigmoid(model1(x))
            p2 = torch.sigmoid(model2(x))

            p = (w1*p1 + w2*p2).cpu().numpy()

            preds.extend(p)
            labels.extend(y.numpy())

    preds_bin = [1 if p>0.5 else 0 for p in preds]

    return {
        "acc": accuracy_score(labels, preds_bin),
        "auc": roc_auc_score(labels, preds),
        "f1": f1_score(labels, preds_bin)
    }

In [ ]:
for w in [0.3, 0.5, 0.7]:
    print(w, ensemble_eval_weighted(w, 1-w))

### **Using a Meta-Learner (Stacking)**

In [ ]:
from sklearn.linear_model import LogisticRegression

meta_X, meta_y = [], []

with torch.no_grad():
    for x,y in loader:
        x = x.to(device)

        p1 = torch.sigmoid(model1(x)).cpu().numpy()
        p2 = torch.sigmoid(model2(x)).cpu().numpy()

        combined = np.hstack([p1, p2])

        meta_X.extend(combined)
        meta_y.extend(y.numpy())

meta_model = LogisticRegression()
meta_model.fit(meta_X, meta_y)

### **Visualizing Performance**

**ROC Curve**

In [ ]:
from sklearn.metrics import roc_curve
import matplotlib.pyplot as plt

fpr, tpr, _ = roc_curve(meta_y, meta_model.predict_proba(meta_X)[:,1])
plt.plot(fpr, tpr)
plt.title("ROC Curve")
plt.show()

**Final Model**

In [ ]:
import joblib
joblib.dump(meta_model, "/content/final_ensemble.pkl")